In [1]:
from __future__ import annotations

# ============================================================
# 0. IMPORTS
# ============================================================

import operator
import os
import re
from datetime import date
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langgraph.checkpoint.postgres import PostgresSaver

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_ollama import ChatOllama

C:\Users\vothi\AppData\Local\Temp\ipykernel_17768\3137389031.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
# ============================================================
# 1. ENVIRONMENT
# ============================================================

load_dotenv()


def get_database_url() -> str:
    """
    Lấy DATABASE_URL từ file .env.

    DATABASE_URL dùng để kết nối PostgreSQL,
    nơi LangGraph lưu checkpoint / state của workflow.
    """

    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. "
            "Please add your Render PostgreSQL External Database URL to .env"
        )

    # PostgreSQL trên Render thường cần SSL.
    # Nếu URL chưa có sslmode thì tự động thêm.
    if "sslmode=" not in database_url:
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    return database_url

In [3]:
# ============================================================
# 2. PYDANTIC SCHEMAS
# ============================================================
#
# Đây là "hợp đồng dữ liệu" giữa LLM và workflow.
#
# Thay vì để LLM trả JSON tự do:
#
#     {"title": "...", "abc": "..."}
#
# ta ép nó phải tuân theo schema:
#
#     Plan
#       └── tasks
#             ├── Task
#             ├── Task
#             └── ...
#
# Đây là nền tảng để dùng:
#
#     llm.with_structured_output(...)
#
# ============================================================


# ------------------------------------------------------------
# 2.1 Task = Công việc cụ thể phải làm để hoàn thành Plan
# ------------------------------------------------------------

# Task
# │
# ├── id                  → Công việc số mấy?
# ├── title               → Tên phần cần viết?
# ├── goal                → Viết phần này để đạt mục tiêu gì?
# ├── bullets             → Cụ thể phải nói những gì để đạt được mục tiêu?
# ├── target_words        → Viết dài khoảng bao nhiêu?
# ├── tags                → Phần này thuộc chủ đề nào?
# ├── requires_research   → Có cần research không?
# ├── requires_citations  → Có cần nguồn trích dẫn không?
# └── requires_code       → Có cần code example không?

class Task(BaseModel):

    id: int

    title: str

    goal: str = Field(
        ...,
        description=(
            "One sentence describing what the reader "
            "should be able to do/understand after this section."
        ),
    )

    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=6,
        description=(
            "3–6 concrete, non-overlapping subpoints "
            "to cover in this section."
        ),
    )

    target_words: int = Field(
        ...,
        description="Target word count for this section (120–550).",
    )

    tags: List[str] = Field(default_factory=list)

    # Section này có cần thông tin từ web không?
    requires_research: bool = False

    # Section này có bắt buộc citation không?
    requires_citations: bool = False

    # Section này có cần code example không?
    requires_code: bool = False

In [4]:
# ------------------------------------------------------------
# 2.2 Plan = Kế hoạch tổng thể cho bài viết
# ------------------------------------------------------------

# Plan
# │
# ├── blog_title   → Bài viết tên gì?
# ├── audience     → Viết cho ai?
# ├── tone         → Viết như thế nào?
# ├── blog_kind    → Viết loại bài gì?
# ├── constraints  → Toàn bài phải tuân thủ gì?
# └── tasks        → Cần làm những section nào?

class Plan(BaseModel):

    blog_title: str

    audience: str

    tone: str

    blog_kind: Literal[
        "explainer",
        "tutorial",
        "news_roundup",
        "comparison",
        "system_design",
    ] = "explainer"

    constraints: List[str] = Field(default_factory=list)

    tasks: List[Task]

In [5]:
# ------------------------------------------------------------
# 2.3 Evidence = một mẩu bằng chứng/thông tin lấy từ một nguồn cụ thể trên Internet
# ------------------------------------------------------------

# EvidenceItem
# │
# ├── title          → Tiêu đề nguồn
# ├── url            → Link nguồn để kiểm chứng/citation
# ├── published_at   → Ngày xuất bản (có thể không có)
# ├── snippet        → Đoạn tóm tắt/nội dung liên quan
# └── source         → Tên tổ chức/website cung cấp nguồn

class EvidenceItem(BaseModel):

    title: str

    url: str

    # Có thể Tavily không trả published date.
    published_at: Optional[str] = None

    snippet: Optional[str] = None

    source: Optional[str] = None

In [6]:
# ------------------------------------------------------------
# 2.4 RouterDecision = quyết định của Router xem có cần research không, research theo kiểu nào và cần search những gì?
# ------------------------------------------------------------

# RouterDecision
# │
# ├── mode
# │      → Chọn cách xử lý:
# │           closed_book → chỉ dùng kiến thức LLM
# │           hybrid      → LLM + research
# │           open_book   → research bên ngoài
# │
# ├── queries
# │      → Các câu truy vấn cần dùng để search
# │
# └── rationale
#        → Giải thích lý do Router chọn mode đó

class RouterDecision(BaseModel):

    # Có cần research không?
    needs_research: bool

    # closed_book / hybrid / open_book
    mode: Literal[
        "closed_book",
        "hybrid",
        "open_book",
    ]

    # Các query gửi cho Tavily
    queries: List[str] = Field(default_factory=list)

In [7]:
# ------------------------------------------------------------
# 2.5 EvidencePack = dùng để đóng gói (pack) danh sách các EvidenceItem
# ------------------------------------------------------------

# EvidencePack
# │
# └── items
#       → Gom nhiều EvidenceItem thành một danh sách
#       → Đóng gói toàn bộ evidence của quá trình research

class EvidencePack(BaseModel):

    evidence: List[EvidenceItem] = Field(
        default_factory=list
    )

In [8]:
# ------------------------------------------------------------
# 2.6 ImageSpec = bản thiết kế cho 1 hình ảnh trong bài blog
# ------------------------------------------------------------

# ImageSpec
# │
# ├── placeholder → Vị trí chèn ảnh trong Markdown
# ├── filename    → Tên file ảnh được lưu
# ├── alt         → Mô tả ảnh cho accessibility
# ├── caption     → Chú thích hiển thị dưới ảnh
# ├── prompt      → Prompt gửi Image Model
# ├── size        → Kích thước ảnh
# └── quality     → Chất lượng ảnh

class ImageSpec(BaseModel):

    # Placeholder xuất hiện trong Markdown
    #
    # Ví dụ:
    #     [[IMAGE_1]]
    #
    placeholder: str = Field(
        ...,
        description="e.g. [[IMAGE_1]]",
    )

    # Tên file image
    filename: str = Field(
        ...,
        description="Save under images/, e.g. qkv_flow.png",
    )

    alt: str

    caption: str

    # Prompt gửi cho image model
    prompt: str = Field(
        ...,
        description="Prompt to send to the image model.",
    )

    size: Literal[
        "1024x1024",
        "1024x1536",
        "1536x1024",
    ] = "1024x1024"

    quality: Literal[
        "low",
        "medium",
        "high",
    ] = "medium"

In [9]:
# ------------------------------------------------------------
# 2.7 GlobalImagePlan = kế hoạch quản lý TOÀN BỘ hình ảnh của bài blog.
# ------------------------------------------------------------

# GlobalImagePlan
# │
# ├── md_with_placeholders → Markdown hoàn chỉnh có [[IMAGE_X]]
# │
# └── images → Danh sách các ảnh cần generate
#       │
#       ├── ImageSpec → Thông tin chi tiết ảnh 1
#       ├── ImageSpec → Thông tin chi tiết ảnh 2
#       └── ...

class GlobalImagePlan(BaseModel):

    # Markdown sau khi LLM chèn [[IMAGE_1]], ...
    md_with_placeholders: str

    # Danh sách image cần generate
    images: List[ImageSpec] = Field(
        default_factory=list
    )

In [10]:
# ============================================================
# 3. LANGGRAPH STATE
# ============================================================
#
# State là "bộ nhớ dùng chung" của workflow.
#
# Có thể hình dung:
#
#                 STATE
#                   │
#       ┌───────────┼────────────┐
#       ▼           ▼            ▼
#    Router      Planner       Worker
#       │           │            │
#       └───────────┼────────────┘
#                   ▼
#                Reducer
#
# ============================================================

class State(TypedDict):

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    topic: str

    # --------------------------------------------------------
    # Router / Research
    # --------------------------------------------------------

    mode: str

    needs_research: bool

    queries: List[str]

    evidence: List[EvidenceItem]

    # --------------------------------------------------------
    # Planner
    # --------------------------------------------------------

    plan: Optional[Plan]

    # --------------------------------------------------------
    # Workers
    # --------------------------------------------------------
    #
    # Mỗi worker trả:
    #
    #     (task_id, section_markdown)
    #
    # operator.add giúp LangGraph MERGE kết quả
    # từ nhiều worker lại.
    #
    # Ví dụ:
    #
    # Worker 1 -> [(1, "...")]
    # Worker 2 -> [(2, "...")]
    # Worker 3 -> [(3, "...")]
    #
    # ↓
    #
    # sections =
    # [
    #     (1, "..."),
    #     (2, "..."),
    #     (3, "...")
    # ]
    #
    # --------------------------------------------------------

    sections: Annotated[
        List[tuple[int, str]],
        operator.add
    ]

    # --------------------------------------------------------
    # Reducer / Images
    # --------------------------------------------------------

    merged_md: str

    md_with_placeholders: str

    image_specs: List[dict]

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    final: str

In [11]:
# ============================================================
# 4. LLM
# ============================================================
#
# Phiên bản này sử dụng Ollama chạy LOCAL.
#
# Model:
#
#     llama3.1:8b
#
# Không cần gửi prompt sang Groq/OpenAI.
#
# ============================================================

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0,
    num_ctx=32768,
)

In [12]:
# ============================================================
# 5. ROUTER
# ============================================================
#
# Router trả lời câu hỏi:
#
#     "Topic này có cần web research không?"
#
# Có 3 mode:
#
# closed_book
#     ↓
# Không cần web.
#
# hybrid
#     ↓
# Có kiến thức nền + một phần thông tin mới.
#
# open_book
#     ↓
# Phụ thuộc mạnh vào thông tin mới nhất.
#
# ============================================================

ROUTER_SYSTEM = """
    You are a routing module for a technical blog planner.

    Decide whether web research is needed BEFORE planning.

    Modes:

    - closed_book (needs_research=false):
    Evergreen topics where correctness does not depend on
    recent facts (concepts, fundamentals).

    - hybrid (needs_research=true):
    Mostly evergreen but needs up-to-date examples/tools/models
    to be useful.

    - open_book (needs_research=true):
    Mostly volatile: weekly roundups, "this week", "latest",
    rankings, pricing, policy/regulation.

    If needs_research=true:

    - Output 3–10 high-signal queries.
    - Queries should be scoped and specific.
    - Avoid generic queries like just "AI" or "LLM".
    - If user asked for "last week/this week/latest",
    reflect that constraint IN THE QUERIES.
"""


def router_node(state: State) -> dict:

    topic = state["topic"]

    # Structured output giúp LLM trả đúng RouterDecision
    decider = llm.with_structured_output(
        RouterDecision
    )

    decision = decider.invoke(
        [
            SystemMessage(
                content=ROUTER_SYSTEM
            ),
            HumanMessage(
                content=f"Topic: {topic}"
            ),
        ]
    )

    print("\n========== ROUTER ==========")
    print(decision.model_dump())

    return {
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries,
    }


# ------------------------------------------------------------
# Router conditional edge
# ------------------------------------------------------------

def route_next(state: State) -> str:

    if state["needs_research"]:
        return "research"

    return "orchestrator"

In [13]:
# ============================================================
# 6. TAVILY RESEARCH
# ============================================================

def _tavily_search(
    query: str,
    max_results: int = 5,
) -> List[dict]:
    """
    Chạy một Tavily search và normalize kết quả.

    Normalize rất quan trọng vì:
    raw result của tool có thể có format khác nhau.

    Ta chuyển tất cả về format thống nhất.
    """

    tool = TavilySearchResults(
        max_results=max_results
    )

    results = tool.invoke(
        {
            "query": query
        }
    )

    normalized: List[dict] = []

    for r in results or []:

        normalized.append(
            {
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": (
                    r.get("content")
                    or r.get("snippet")
                    or ""
                ),
                "published_at": (
                    r.get("published_date")
                    or r.get("published_at")
                ),
                "source": r.get("source"),
            }
        )

    return normalized

In [14]:
# ============================================================
# 7. RESEARCH SYNTHESIZER
# ============================================================

RESEARCH_SYSTEM = """
    You are a research synthesizer for technical writing.

    Given raw web search results, produce a deduplicated
    list of EvidenceItem objects.

    Rules:

    - Only include items with a non-empty url.
    - Prefer relevant + authoritative sources.
    - If a published date is explicitly present,
    keep it as YYYY-MM-DD.
    - If missing or unclear, set published_at=null.
    - Do NOT guess dates.
    - Keep snippets short.
    - Deduplicate by URL.
"""


def research_node(state: State) -> dict:

    queries = (
        state.get("queries", [])
        or []
    )

    max_results = 6

    raw_results: List[dict] = []

    # --------------------------------------------------------
    # Chạy từng query
    # --------------------------------------------------------

    for query in queries:

        raw_results.extend(
            _tavily_search(
                query,
                max_results=max_results,
            )
        )

    # Không có kết quả
    if not raw_results:

        print("\n⚠️ No research results.")

        return {
            "evidence": []
        }

    print(
        f"\n🔎 Raw research results: "
        f"{len(raw_results)}"
    )

    # --------------------------------------------------------
    # LLM tổng hợp evidence
    # --------------------------------------------------------

    extractor = llm.with_structured_output(
        EvidencePack
    )

    pack = extractor.invoke(
        [
            SystemMessage(
                content=RESEARCH_SYSTEM
            ),
            HumanMessage(
                content=(
                    f"Raw results:\n"
                    f"{raw_results}"
                )
            ),
        ]
    )

    # --------------------------------------------------------
    # Deduplicate theo URL
    # --------------------------------------------------------

    dedup = {}

    for evidence in pack.evidence:

        if evidence.url:
            dedup[evidence.url] = evidence

    evidence = list(
        dedup.values()
    )

    print(
        f"✅ Evidence after dedup: "
        f"{len(evidence)}"
    )

    return {
        "evidence": evidence
    }

In [15]:
# ============================================================
# 8. ORCHESTRATOR / PLANNER
# ============================================================
#
# Router:
#
#     "Có cần research không?"
#
# Research:
#
#     "Đây là evidence."
#
# Orchestrator:
#
#     "Dựa vào topic + mode + evidence,
#      hãy lập kế hoạch."
#
# ============================================================

ORCH_SYSTEM = """
    You are a senior technical writer and developer advocate.

    Your job is to produce a highly actionable outline
    for a technical blog post.

    Hard requirements:

    - Create 5–9 sections (tasks).
    - Each task must include:
    1) goal
    2) 3–6 bullets
    3) target word count (120–550)

    Quality bar:

    - Assume the reader is a developer.
    - Use correct terminology.
    - Bullets must be actionable:
    build/compare/measure/verify/debug.

    Ensure the overall plan includes at least 2 of:

    - minimal code sketch / MWE
    - edge cases / failure modes
    - performance/cost considerations
    - security/privacy considerations
    - debugging/observability tips

    Grounding rules:

    - closed_book:
    keep it evergreen.

    - hybrid:
    use evidence for up-to-date examples.
    Mark fresh sections with:
        requires_research=True
        requires_citations=True

    - open_book:
    set blog_kind="news_roundup".
    Every section should summarize events + implications.
    Do NOT create tutorial sections unless explicitly requested.

    If evidence is insufficient:
    transparently say "insufficient sources".

    Output must strictly match the Plan schema.
"""


def orchestrator_node(state: State) -> dict:

    planner = llm.with_structured_output(
        Plan
    )

    evidence = state.get(
        "evidence",
        []
    )

    mode = state.get(
        "mode",
        "closed_book"
    )

    evidence_data = [
        e.model_dump()
        for e in evidence
    ]

    plan = planner.invoke(
        [
            SystemMessage(
                content=ORCH_SYSTEM
            ),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence "
                    f"(ONLY use for fresh claims; "
                    f"may be empty):\n"
                    f"{evidence_data[:16]}"
                )
            ),
        ]
    )

    print("\n========== PLAN ==========")
    print(
        plan.model_dump_json(
            indent=2
        )
    )

    return {
        "plan": plan
    }

In [16]:
# ============================================================
# 9. FAN-OUT
# ============================================================
#
# Đây là một phần rất quan trọng của LangGraph.
#
# Plan:
#
#     Task 1
#     Task 2
#     Task 3
#     Task 4
#
# fanout biến nó thành:
#
#     Worker(Task 1)
#     Worker(Task 2)
#     Worker(Task 3)
#     Worker(Task 4)
#
# Các worker có thể chạy song song.
#
# ============================================================

def fanout(state: State):

    plan = state["plan"]

    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),

                "topic": state["topic"],

                "mode": state["mode"],

                "plan": plan.model_dump(),

                "evidence": [
                    e.model_dump()
                    for e in state.get(
                        "evidence",
                        []
                    )
                ],
            },
        )

        for task in plan.tasks
    ]

In [17]:
# ============================================================
# 10. WORKER
# ============================================================
#
# Mỗi worker chỉ chịu trách nhiệm:
#
#     1 section
#
# Không viết toàn bộ blog.
#
# Đây là tư tưởng:
#
#     Planner → phân chia công việc
#     Worker  → thực thi từng phần
#
# ============================================================

WORKER_SYSTEM = """
You are a senior technical writer and developer advocate.

Write ONE section of a technical blog post in Markdown.

Hard constraints:

- Follow the provided Goal.
- Cover ALL Bullets in order.
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown.
- Do NOT output blog title H1.
- Start with:
  ## <Section Title>

Scope guard:

- If blog_kind == "news_roundup":
  do NOT turn this into a tutorial.
- Focus on summarizing events and implications.

Grounding policy:

- If mode == open_book:
  specific event/company/model/funding/policy claims
  MUST be supported by provided Evidence URLs.

- For each event claim:
  attach a Markdown source link.

- Only use URLs provided in Evidence.

- If information is not supported:
  write:
  "Not found in provided sources."

- If requires_citations == true:
  cite Evidence URLs.

- Evergreen reasoning is okay without citations
  unless requires_citations=true.

Code:

- If requires_code == true:
  include at least one minimal,
  correct code snippet.

Style:

- Short paragraphs.
- Bullets where useful.
- Code fences for code.
- Avoid fluff/marketing.
- Be precise and implementation-oriented.
"""


def worker_node(payload: dict) -> dict:

    # --------------------------------------------------------
    # Deserialize payload
    #
    # Send() truyền dictionary,
    # nên ta convert ngược về Pydantic model.
    # --------------------------------------------------------

    task = Task(
        **payload["task"]
    )

    plan = Plan(
        **payload["plan"]
    )

    evidence = [
        EvidenceItem(**e)
        for e in payload.get(
            "evidence",
            []
        )
    ]

    topic = payload["topic"]

    mode = payload.get(
        "mode",
        "closed_book"
    )

    # --------------------------------------------------------
    # Format bullets
    # --------------------------------------------------------

    bullets_text = (
        "\n- "
        + "\n- ".join(task.bullets)
    )

    # --------------------------------------------------------
    # Format evidence
    # --------------------------------------------------------

    evidence_text = ""

    if evidence:

        evidence_text = "\n".join(
            (
                f"- {e.title} | "
                f"{e.url} | "
                f"{e.published_at or 'date:unknown'}"
            ).strip()

            for e in evidence[:20]
        )

    # --------------------------------------------------------
    # Generate section
    # --------------------------------------------------------

    response = llm.invoke(
        [
            SystemMessage(
                content=WORKER_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {plan.constraints}\n"
                    f"Topic: {topic}\n"
                    f"Mode: {mode}\n\n"

                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: "
                    f"{task.requires_research}\n"
                    f"requires_citations: "
                    f"{task.requires_citations}\n"
                    f"requires_code: "
                    f"{task.requires_code}\n"

                    f"Bullets:"
                    f"{bullets_text}\n\n"

                    "Evidence "
                    "(ONLY use these URLs when citing):\n"
                    f"{evidence_text}\n"
                )
            ),
        ]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # AIMessage.content thường là str,
    # nhưng một số provider có thể trả list.
    #
    # Ta normalize để worker không bị:
    #
    #     AttributeError:
    #     'list' object has no attribute 'strip'
    #
    # --------------------------------------------------------

    content = response.content

    if isinstance(content, list):

        section_md = "".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    else:

        section_md = content

    section_md = section_md.strip()

    print(
        f"\n✍️ Worker finished: "
        f"Task {task.id} - {task.title}"
    )

    # --------------------------------------------------------
    # QUAN TRỌNG:
    #
    # task.id được trả về cùng section.
    #
    # Reducer sau đó sort theo task.id
    # để đảm bảo thứ tự blog không bị đảo.
    # --------------------------------------------------------

    return {
        "sections": [
            (
                task.id,
                section_md
            )
        ]
    }

In [18]:
# ============================================================
# 11. REDUCER SUBGRAPH
# ============================================================
#
# Đây là điểm nâng cấp khá lớn của project.
#
# Reducer không còn chỉ là:
#
#     worker → reducer → END
#
# Mà trở thành một SUBGRAPH:
#
#                 reducer
#                    │
#                    ▼
#              merge_content
#                    │
#                    ▼
#              decide_images
#                    │
#                    ▼
#        generate_and_place_images
#
# ============================================================


# ============================================================
# 11.1 MERGE CONTENT
# ============================================================

def merge_content(state: State) -> dict:

    plan = state["plan"]

    # --------------------------------------------------------
    # Worker chạy song song nên thứ tự result
    # không nhất thiết giống Task ID.
    #
    # Ví dụ:
    #
    # [(3, "..."), (1, "..."), (2, "...")]
    #
    # Sort lại:
    #
    # [(1, "..."), (2, "..."), (3, "...")]
    # --------------------------------------------------------

    ordered_sections = [
        md
        for _, md in sorted(
            state["sections"],
            key=lambda x: x[0]
        )
    ]

    body = "\n\n".join(
        ordered_sections
    ).strip()

    merged_md = (
        f"# {plan.blog_title}\n\n"
        f"{body}\n"
    )

    return {
        "merged_md": merged_md
    }

In [19]:
# ============================================================
# 11.2 DECIDE IMAGES
# ============================================================

DECIDE_IMAGES_SYSTEM = """
You are an expert technical editor.

Decide if images/diagrams are needed for THIS blog.

Rules:

- Maximum 3 images total.
- Each image must materially improve understanding.
- Prefer diagrams/flows/technical visuals.
- Avoid decorative images.
- Insert placeholders exactly:
  [[IMAGE_1]]
  [[IMAGE_2]]
  [[IMAGE_3]]

If no images are needed:

    md_with_placeholders = input

    images = []

Return strictly GlobalImagePlan.
"""


def decide_images(state: State) -> dict:

    planner = llm.with_structured_output(
        GlobalImagePlan
    )

    merged_md = state["merged_md"]

    plan = state["plan"]

    assert plan is not None

    image_plan = planner.invoke(
        [
            SystemMessage(
                content=DECIDE_IMAGES_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Blog kind: "
                    f"{plan.blog_kind}\n"
                    f"Topic: "
                    f"{state['topic']}\n\n"

                    "Insert placeholders + "
                    "propose image prompts.\n\n"

                    f"{merged_md}"
                )
            ),
        ]
    )

    print("\n========== IMAGE PLAN ==========")

    print(
        image_plan.model_dump_json(
            indent=2
        )
    )

    return {
        "md_with_placeholders": (
            image_plan.md_with_placeholders
        ),

        "image_specs": [
            image.model_dump()
            for image in image_plan.images
        ],
    }

In [20]:
# ============================================================
# 11.3 GEMINI IMAGE GENERATION
# ============================================================
#
# Lưu ý:
#
# Text model:
#     Ollama llama3.1:8b
#
# Image model:
#     Gemini
#
# Tức là project đang dùng TWO providers:
#
#     Ollama → text
#     Gemini → image
#
# ============================================================

def _gemini_generate_image_bytes(
    prompt: str,
) -> bytes:
    """
    Generate image bằng Gemini.

    Requires:
        pip install google-genai

    Environment:
        GOOGLE_API_KEY
    """

    from google import genai
    from google.genai import types

    api_key = os.environ.get(
        "GOOGLE_API_KEY"
    )

    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set."
        )

    client = genai.Client(
        api_key=api_key
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash-image",

        contents=prompt,

        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],

            safety_settings=[
                types.SafetySetting(
                    category=(
                        "HARM_CATEGORY_DANGEROUS_CONTENT"
                    ),
                    threshold="BLOCK_ONLY_HIGH",
                )
            ],
        ),
    )

    # --------------------------------------------------------
    # SDK có thể expose parts ở nhiều vị trí.
    # --------------------------------------------------------

    parts = getattr(
        response,
        "parts",
        None,
    )

    if (
        not parts
        and getattr(
            response,
            "candidates",
            None,
        )
    ):

        try:

            parts = (
                response
                .candidates[0]
                .content
                .parts
            )

        except Exception:

            parts = None

    if not parts:

        raise RuntimeError(
            "No image content returned "
            "(safety/quota/SDK change)."
        )

    # --------------------------------------------------------
    # Tìm inline image data
    # --------------------------------------------------------

    for part in parts:

        inline = getattr(
            part,
            "inline_data",
            None,
        )

        if (
            inline
            and getattr(
                inline,
                "data",
                None,
            )
        ):

            return inline.data

    raise RuntimeError(
        "No inline image bytes found "
        "in response."
    )

In [21]:
# ============================================================
# 11.4 GENERATE + PLACE IMAGES
# ============================================================

def generate_and_place_images(
    state: State,
) -> dict:

    plan = state["plan"]

    assert plan is not None

    md = (
        state.get(
            "md_with_placeholders"
        )
        or state["merged_md"]
    )

    image_specs = (
        state.get(
            "image_specs",
            []
        )
        or []
    )

    # --------------------------------------------------------
    # Nếu LLM quyết định không cần ảnh
    # --------------------------------------------------------

    if not image_specs:

        filename = (
            safe_filename(
                plan.blog_title
            )
            + ".md"
        )

        Path(filename).write_text(
            md,
            encoding="utf-8",
        )

        print(
            f"\n📄 Markdown saved: "
            f"{Path(filename).resolve()}"
        )

        return {
            "final": md
        }

    # --------------------------------------------------------
    # Tạo thư mục images/
    # --------------------------------------------------------

    images_dir = Path(
        "images"
    )

    images_dir.mkdir(
        exist_ok=True
    )

    # --------------------------------------------------------
    # Generate từng image
    # --------------------------------------------------------

    for spec in image_specs:

        placeholder = (
            spec["placeholder"]
        )

        filename = (
            spec["filename"]
        )

        out_path = (
            images_dir / filename
        )

        # ----------------------------------------------------
        # Chỉ generate nếu image chưa tồn tại
        # ----------------------------------------------------

        if not out_path.exists():

            try:

                img_bytes = (
                    _gemini_generate_image_bytes(
                        spec["prompt"]
                    )
                )

                out_path.write_bytes(
                    img_bytes
                )

                print(
                    f"🖼️ Image saved: "
                    f"{out_path}"
                )

            except Exception as e:

                # ------------------------------------------------
                # Graceful fallback:
                #
                # Nếu image generation fail,
                # blog vẫn được tạo.
                #
                # ------------------------------------------------

                prompt_block = (
                    "> **[IMAGE GENERATION FAILED]** "
                    f"{spec.get('caption', '')}\n"
                    ">\n"
                    f"> **Alt:** "
                    f"{spec.get('alt', '')}\n"
                    ">\n"
                    f"> **Prompt:** "
                    f"{spec.get('prompt', '')}\n"
                    ">\n"
                    f"> **Error:** {e}\n"
                )

                md = md.replace(
                    placeholder,
                    prompt_block,
                )

                continue

        # ----------------------------------------------------
        # Thay placeholder bằng Markdown image
        # ----------------------------------------------------

        img_md = (
            f"![{spec['alt']}]"
            f"(images/{filename})\n"
            f"*{spec['caption']}*"
        )

        md = md.replace(
            placeholder,
            img_md,
        )

    # --------------------------------------------------------
    # Save final Markdown
    # --------------------------------------------------------

    filename = (
        safe_filename(
            plan.blog_title
        )
        + ".md"
    )

    output_path = Path(
        filename
    )

    output_path.write_text(
        md,
        encoding="utf-8",
    )

    print(
        f"\n📄 Final Markdown saved:"
        f"\n{output_path.resolve()}"
    )

    return {
        "final": md
    }

In [22]:
# ============================================================
# 12. SAFE FILENAME
# ============================================================
#
# Đây là phần mình cố tình thêm.
#
# Trước đây title có thể là:
#
#     Understanding Machine Learning:
#     A Beginner's Roadmap
#
# Windows không cho phép ':' trong filename.
#
# Vì vậy phải sanitize title trước khi tạo file.
#
# ============================================================

def safe_filename(
    title: str,
) -> str:

    # Xóa ký tự không hợp lệ trên Windows
    filename = re.sub(
        r'[<>:"/\\|?*]',
        "",
        title,
    )

    # Thay nhiều khoảng trắng bằng _
    filename = re.sub(
        r"\s+",
        "_",
        filename,
    )

    return filename.lower()

In [23]:
# ============================================================
# 13. BUILD REDUCER SUBGRAPH
# ============================================================
#
# Subgraph:
#
# START
#   ↓
# merge_content
#   ↓
# decide_images
#   ↓
# generate_and_place_images
#   ↓
# END
#
# ============================================================

reducer_graph = StateGraph(
    State
)

reducer_graph.add_node(
    "merge_content",
    merge_content,
)

reducer_graph.add_node(
    "decide_images",
    decide_images,
)

reducer_graph.add_node(
    "generate_and_place_images",
    generate_and_place_images,
)

reducer_graph.add_edge(
    START,
    "merge_content",
)

reducer_graph.add_edge(
    "merge_content",
    "decide_images",
)

reducer_graph.add_edge(
    "decide_images",
    "generate_and_place_images",
)

reducer_graph.add_edge(
    "generate_and_place_images",
    END,
)

reducer_subgraph = (
    reducer_graph.compile()
)

In [24]:
# ============================================================
# 14. MAIN GRAPH
# ============================================================
#
# Đây là toàn bộ Agentic Writer:
#
#
#                     START
#                       │
#                       ▼
#                    ROUTER
#                  /         \
#                 /           \
#                ▼             ▼
#           RESEARCH       ORCHESTRATOR
#                │             │
#                └──────┬──────┘
#                       ▼
#                  ORCHESTRATOR
#                       │
#                     FANOUT
#                  /    |    \
#                 ▼     ▼     ▼
#              Worker Worker Worker
#                 \     |     /
#                  \    |    /
#                    REDUCER
#                       │
#                ┌──────┴──────┐
#                ▼             ▼
#             Merge         Images
#                │             │
#                └──────┬──────┘
#                       ▼
#                      END
#
# ============================================================

g = StateGraph(
    State
)

# ------------------------------------------------------------
# Main nodes
# ------------------------------------------------------------

g.add_node(
    "router",
    router_node,
)

g.add_node(
    "research",
    research_node,
)

g.add_node(
    "orchestrator",
    orchestrator_node,
)

g.add_node(
    "worker",
    worker_node,
)

# Subgraph được gắn như một node
g.add_node(
    "reducer",
    reducer_subgraph,
)


# ------------------------------------------------------------
# Main edges
# ------------------------------------------------------------

g.add_edge(
    START,
    "router",
)

# Router quyết định:
#
# research
#     hoặc
#
# orchestrator
#
g.add_conditional_edges(
    "router",
    route_next,
    {
        "research": "research",
        "orchestrator": "orchestrator",
    },
)

# Research xong → Planner
g.add_edge(
    "research",
    "orchestrator",
)

# Planner → Fan-out Workers
g.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"],
)

# Worker → Reducer Subgraph
g.add_edge(
    "worker",
    "reducer",
)

# Reducer → END
g.add_edge(
    "reducer",
    END,
)

In [25]:
# ============================================================
# 15. POSTGRES CHECKPOINTER
# ============================================================
#
# Đây là nâng cấp lớn so với InMemorySaver.
#
# InMemorySaver:
#
#     RAM
#      ↓
#     restart app
#      ↓
#     mất state
#
#
# PostgreSQL:
#
#     LangGraph
#         ↓
#     PostgreSQL
#         ↓
#     restart app
#         ↓
#     checkpoint vẫn còn
#
# ============================================================

DATABASE_URL = (
    get_database_url()
)

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row,
)

checkpointer = PostgresSaver(
    _conn
)

# Tạo các bảng checkpoint nếu chưa tồn tại
checkpointer.setup()


# ------------------------------------------------------------
# Compile final application
# ------------------------------------------------------------

app = g.compile(
    checkpointer=checkpointer
)

In [26]:
# ============================================================
# 16. CONFIG
# ============================================================
#
# thread_id dùng để LangGraph biết:
#
#     "Đây là conversation / workflow nào?"
#
# PostgreSQL sẽ lưu checkpoint dựa trên thread này.
#
# ============================================================

config = {
    "configurable": {
        "thread_id": "test_thread_id_2"
    }
}

In [27]:
# ============================================================
# 17. RUNNER
# ============================================================

def run(
    topic: str,
    as_of: Optional[str] = None,
):

    # Nếu caller không truyền ngày,
    # lấy ngày hiện tại.
    if as_of is None:
        as_of = date.today().isoformat()

    # --------------------------------------------------------
    # IMPORTANT
    #
    # State hiện tại không dùng as_of.
    #
    # Vì vậy KHÔNG truyền as_of vào app.invoke()
    # cho tới khi bạn thực sự thêm nó vào State.
    # --------------------------------------------------------

    initial_state = {

        "topic": topic,

        "mode": "",

        "needs_research": False,

        "queries": [],

        "evidence": [],

        "plan": None,

        "sections": [],

        "merged_md": "",

        "md_with_placeholders": "",

        "image_specs": [],

        "final": "",
    }

    out = app.invoke(
        initial_state,
        config=config,
    )

    return out

In [28]:
# ============================================================
# 18. TEST
# ============================================================

result = run(
    "Self Attention in Transformer Architecture"
)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

print(
    result.get(
        "final",
        ""
    )
)


========== ROUTER ==========
{'needs_research': False, 'mode': 'closed_book', 'queries': []}

========== PLAN ==========
{
  "blog_title": "Unlocking the Power of Self-Attention in Transformer Architecture",
  "audience": "developers",
  "tone": "technical",
  "blog_kind": "explainer",
  "constraints": [],
  "tasks": [
    {
      "id": 1,
      "title": "Understanding Self-Attention",
      "goal": "Explain the concept of self-attention in the transformer architecture",
      "bullets": [
        "Describe how self-attention allows the model to weigh the importance of different input elements",
        "Explain how self-attention is used in conjunction with other transformer components",
        "Provide a high-level overview of how self-attention is computed"
      ],
      "target_words": 120,
      "tags": [],
      "requires_research": false,
      "requires_citations": false,
      "requires_code": false
    },
    {
      "id": 2,
      "title": "Self-Attention Mechanism",
    